# SAFE-XAI: Explainable and Fair Student Performance Prediction
## Complete Jupyter Notebook

**What this notebook does**
1. Loads `rs.csv` if it is present in the notebook folder; otherwise creates a clearly labeled synthetic example dataset.
2. Inspects and preprocesses data.
3. Trains and compares Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting.
4. Evaluates classification using accuracy, precision, recall, F1, ROC-AUC, confusion matrix, and ROC curve.
5. Explains predictions using permutation importance and, when installed, SHAP and LIME.
6. Includes example fairness and explanation-stability analyses.
7. Saves model metrics, predictions, plots, and a fitted model in `outputs/`.

**Important:** Synthetic data is for software testing only and must not be presented as real research evidence. Results are not guaranteed and must be calculated by running the notebook on your actual dataset. This is a research prototype, not a validated student decision-making system. Do not use sensitive personal attributes to make consequential decisions without appropriate governance and review.


## 1. Install dependencies (run once if needed)
If imports fail, uncomment and run the following cell. Internet access may be required.

In [ ]:
# Uncomment if needed:
# %pip install pandas numpy matplotlib scikit-learn joblib shap lime openpyxl


In [ ]:
from pathlib import Path
import warnings, json, sys, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Output folder:", OUTPUT_DIR.resolve())


## 2. Load the dataset
Expected target column: `Performance` or `Pass/Fail` (case-insensitive). If neither is found, the notebook will stop and ask you to set `TARGET_COLUMN` explicitly. If `rs.csv` is absent, a synthetic dataset is created so the code can be tested end-to-end.

In [ ]:
DATA_PATH = Path("rs.csv")  # Change this if your CSV has another filename
TARGET_COLUMN = None       # Example: "Pass/Fail" or "Performance"

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE = f"User dataset: {DATA_PATH}"
    print("Loaded:", DATA_PATH.resolve())
else:
    print("rs.csv not found. Creating a synthetic demonstration dataset.")
    n = 1200
    attendance = np.clip(np.random.normal(78, 12, n), 25, 100)
    assignment = np.clip(np.random.normal(72, 15, n), 0, 100)
    internal = np.clip(np.random.normal(68, 14, n), 0, 100)
    study_hours = np.clip(np.random.normal(2.8, 1.3, n), 0, 8)
    previous_gpa = np.clip(np.random.normal(6.8, 1.2, n), 0, 10)
    quiz = np.clip(np.random.normal(70, 16, n), 0, 100)
    stress = np.random.choice(["Low", "Medium", "High"], n, p=[.3, .5, .2])
    learning_style = np.random.choice(["Visual", "Reading", "Practical"], n)
    age = np.random.randint(18, 25, n)
    score = (0.025*attendance + 0.018*assignment + 0.022*internal +
             0.9*study_hours + 1.5*previous_gpa + 0.012*quiz -
             np.where(stress == "High", 1.0, np.where(stress == "Medium", .3, 0)) +
             np.random.normal(0, 2.0, n))
    threshold = np.median(score)
    df = pd.DataFrame({
        "Attendance": attendance, "Assignment Score": assignment,
        "Internal Marks": internal, "Study Hours": study_hours,
        "Previous GPA": previous_gpa, "Quiz Score": quiz,
        "Stress Level": stress, "Learning Style": learning_style, "Age": age,
        "Performance": np.where(score >= threshold, "Pass", "Fail")
    })
    DATA_SOURCE = "Synthetic demonstration data (not real student data)"

print("Data source:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())


## 3. Inspect and clean data

In [ ]:
print("Column names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)


In [ ]:
# Choose target automatically only when a common target name is present.
if TARGET_COLUMN is None:
    target_candidates = ["Performance", "Pass/Fail", "Pass_Fail", "Target", "Result", "Outcome"]
    lookup = {str(c).strip().lower(): c for c in df.columns}
    found = next((lookup[name.lower()] for name in target_candidates if name.lower() in lookup), None)
    if found is None:
        raise ValueError(
            "Target column not found. Set TARGET_COLUMN to the exact target column name, "
            "then rerun this cell."
        )
    TARGET_COLUMN = found

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"TARGET_COLUMN={TARGET_COLUMN!r} is not in the dataset.")

# Remove rows where the target is missing; keep other missing values for imputation.
df = df.dropna(subset=[TARGET_COLUMN]).copy()
X = df.drop(columns=[TARGET_COLUMN]).copy()
y_raw = df[TARGET_COLUMN].astype(str).str.strip()

# Remove obvious ID columns from model inputs to reduce accidental leakage.
id_like = [c for c in X.columns if str(c).strip().lower() in {"id", "student_id", "student id", "roll_no", "roll number"}]
if id_like:
    print("Dropping identifier columns:", id_like)
    X = X.drop(columns=id_like)

# Encode target labels into integers.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = [str(x) for x in label_encoder.classes_]
if len(class_names) < 2:
    raise ValueError("The target must contain at least two classes.")

print("Target:", TARGET_COLUMN)
print("Classes:", class_names)
print("Feature matrix:", X.shape)
display(pd.Series(y_raw).value_counts().rename_axis("class").to_frame("count"))


## 4. Exploratory data analysis
The following plots are descriptive. If this is a synthetic dataset, they describe only the synthetic example.

In [ ]:
plt.figure(figsize=(7, 4))
y_raw.value_counts().plot(kind="bar")
plt.title(f"Target class distribution: {TARGET_COLUMN}")
plt.xlabel("Class"); plt.ylabel("Count"); plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "target_distribution.png", dpi=160)
plt.show()

numeric_cols_all = X.select_dtypes(include=np.number).columns.tolist()
if numeric_cols_all:
    X[numeric_cols_all].hist(figsize=(14, 9), bins=20)
    plt.suptitle("Numeric feature distributions")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "numeric_distributions.png", dpi=160)
    plt.show()

if len(numeric_cols_all) >= 2:
    corr = X[numeric_cols_all].corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Numeric feature correlation matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=160)
    plt.show()


## 5. Train/test split and preprocessing
Preprocessing is fitted only on the training split to avoid leakage. For a fair final study, use cross-validation and keep a final test set untouched until model selection is complete.

In [ ]:
# Stratification is useful when every class has enough examples.
class_counts = pd.Series(y).value_counts()
stratify_y = y if class_counts.min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=stratify_y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder="drop")

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 6. Train and compare baseline models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE, max_depth=8),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

fitted_models = {}
metric_rows = []
for model_name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }
    if hasattr(pipe, "predict_proba") and len(np.unique(y_test)) == 2:
        try:
            row["ROC_AUC"] = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
        except Exception:
            row["ROC_AUC"] = np.nan
    else:
        row["ROC_AUC"] = np.nan
    metric_rows.append(row)
    fitted_models[model_name] = pipe

metrics_df = pd.DataFrame(metric_rows).sort_values("F1_macro", ascending=False)
display(metrics_df.style.format(precision=3))
metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

best_model_name = metrics_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print("Selected model by test-set macro F1 (exploratory only):", best_model_name)
print("For publication, select models using training-only cross-validation, then evaluate once on a held-out test set.")


## 7. Evaluate the selected model

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(values_format="d")
plt.title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160)
plt.show()

if len(class_names) == 2 and hasattr(best_model, "predict_proba"):
    try:
        RocCurveDisplay.from_estimator(best_model, X_test, y_test)
        plt.title(f"ROC curve — {best_model_name}")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "roc_curve.png", dpi=160)
        plt.show()
    except Exception as e:
        print("ROC curve unavailable:", e)


## 8. Permutation feature importance
Permutation importance works with the full pipeline and original input columns. It measures how much the chosen evaluation score changes when a feature is shuffled; it is not a causal explanation.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE,
    scoring="f1_macro", n_jobs=-1
)
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)
display(importance_df)
importance_df.to_csv(OUTPUT_DIR / "permutation_feature_importance.csv", index=False)

plot_df = importance_df.head(20).sort_values("Importance_Mean")
plt.figure(figsize=(9, max(4, len(plot_df)*0.32)))
plt.barh(plot_df["Feature"].astype(str), plot_df["Importance_Mean"], xerr=plot_df["Importance_Std"])
plt.xlabel("Decrease in macro F1 after permutation")
plt.title("Permutation feature importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "permutation_importance.png", dpi=180)
plt.show()


## 9. Optional SHAP explanations
SHAP APIs can differ by version and model type. This cell attempts a model-agnostic KernelExplainer on a small sample; it may take time. If SHAP is not installed or errors occur, the notebook continues.

In [ ]:
try:
    import shap
    print("SHAP version:", shap.__version__)
    # Use a small sample to keep KernelExplainer practical.
    background_raw = X_train.sample(min(30, len(X_train)), random_state=RANDOM_STATE)
    explain_raw = X_test.sample(min(20, len(X_test)), random_state=RANDOM_STATE)

    def positive_probability(arr):
        # KernelExplainer supplies numpy arrays; restore original column names.
        frame = pd.DataFrame(arr, columns=X_train.columns)
        probs = best_model.predict_proba(frame)
        if probs.shape[1] == 2:
            return probs[:, 1]
        return probs.max(axis=1)

    background_array = background_raw.to_numpy()
    explain_array = explain_raw.to_numpy()
    explainer = shap.KernelExplainer(positive_probability, background_array)
    shap_values = explainer.shap_values(explain_array, nsamples=100)
    vals = np.asarray(shap_values)
    if vals.ndim == 3:
        vals = vals[..., -1] if vals.shape[-1] == len(class_names) else vals[:, :, 0]
    if vals.ndim == 1:
        vals = vals.reshape(1, -1)
    shap.summary_plot(vals, explain_raw, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_summary.png", dpi=180, bbox_inches="tight")
    plt.show()
except ImportError:
    print("SHAP is not installed. Run the install cell if you want SHAP explanations.")
except Exception as e:
    print("SHAP section could not complete with this dataset/version:", repr(e))


## 10. Optional LIME explanations
LIME is model-agnostic. Categorical and mixed-type data may require additional feature configuration; this example uses a small, sampled explanation and is intended as a starting point.

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
    # Encode the training inputs into numeric features for LIME. This generic setup
    # is best suited to numeric-only datasets; for mixed data, configure categorical
    # feature indices/names explicitly before interpreting results.
    if len(categorical_features) == 0:
        lime_explainer = LimeTabularExplainer(
            X_train.to_numpy(dtype=float),
            feature_names=X_train.columns.tolist(),
            class_names=class_names,
            mode="classification",
            discretize_continuous=True,
            random_state=RANDOM_STATE
        )
        row_index = 0
        exp = lime_explainer.explain_instance(
            X_test.iloc[row_index].to_numpy(dtype=float),
            lambda a: best_model.predict_proba(pd.DataFrame(a, columns=X_train.columns)),
            num_features=min(10, X_train.shape[1])
        )
        display(pd.DataFrame(exp.as_list(), columns=["Feature rule", "Contribution"]))
        exp.save_to_file(str(OUTPUT_DIR / "lime_explanation.html"))
        fig = exp.as_pyplot_figure()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "lime_explanation.png", dpi=180, bbox_inches="tight")
        plt.show()
    else:
        print("Mixed numeric/categorical data detected. For valid LIME explanations, configure categorical feature indices and names; see LIME documentation.")
except ImportError:
    print("LIME is not installed. Run the install cell if you want LIME explanations.")
except Exception as e:
    print("LIME section could not complete:", repr(e))


## 11. Fairness audit (illustrative)
Choose a group column that is present in the dataset, such as `Gender`. This cell reports group-level accuracy and positive prediction rates. These descriptive metrics do not establish whether a model is legally or ethically fair. Check sample sizes, target definitions, context, and appropriate fairness criteria before drawing conclusions.

In [ ]:
GROUP_COLUMN = "Gender"  # Change to a column in your dataset, or set to None to skip

if GROUP_COLUMN and GROUP_COLUMN in df.columns:
    group_test = X_test[GROUP_COLUMN].astype(str).reset_index(drop=True)
    y_test_s = pd.Series(y_test).reset_index(drop=True)
    y_pred_s = pd.Series(y_pred).reset_index(drop=True)
    fairness_rows = []
    for group in sorted(group_test.unique()):
        mask = group_test.eq(group).to_numpy()
        if mask.sum() == 0:
            continue
        fairness_rows.append({
            "Group": group,
            "N": int(mask.sum()),
            "Accuracy": accuracy_score(y_test_s[mask], y_pred_s[mask]),
            "Positive_prediction_rate": float(np.mean(y_pred_s[mask] == (1 if len(class_names)==2 else y_pred_s.mode()[0])))
        })
    fairness_df = pd.DataFrame(fairness_rows)
    display(fairness_df)
    fairness_df.to_csv(OUTPUT_DIR / "group_fairness_audit.csv", index=False)
else:
    print(f"Group column {GROUP_COLUMN!r} not found. Set GROUP_COLUMN to a suitable feature to run this audit.")


## 12. Explanation stability using repeated train/test perturbations
This practical approximation checks whether permutation importance rankings change across repeated random seeds. It is not the same as stability of SHAP/LIME explanations for individual cases; label it accurately in any report.

In [ ]:
stability_rows = []
for seed in [11, 22, 33, 44, 55]:
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=seed,
        stratify=y if pd.Series(y).value_counts().min() >= 2 else None
    )
    temp_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=150, class_weight="balanced",
                                         random_state=seed, n_jobs=-1))
    ])
    temp_model.fit(Xtr, ytr)
    pi = permutation_importance(temp_model, Xte, yte, n_repeats=4,
                                random_state=seed, scoring="f1_macro", n_jobs=-1)
    order = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
    stability_rows.append(order.rank(ascending=False).rename(f"seed_{seed}"))

rank_df = pd.concat(stability_rows, axis=1)
rank_corr = rank_df.corr(method="spearman")
display(rank_corr)
rank_corr.to_csv(OUTPUT_DIR / "importance_rank_stability_spearman.csv")

plt.figure(figsize=(7, 5))
plt.imshow(rank_corr, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Spearman correlation of feature ranks")
plt.xticks(range(len(rank_corr.columns)), rank_corr.columns, rotation=45)
plt.yticks(range(len(rank_corr.index)), rank_corr.index)
plt.title("Permutation-importance rank stability across random seeds")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "importance_rank_stability.png", dpi=180)
plt.show()


## 13. Predict a new student
Edit the example values to match your dataset's input columns. This cell checks column names and reports the predicted class and probabilities. Do not interpret predictions as definitive decisions about a student.

In [ ]:
# Example: start from one test row so feature names/types match your dataset.
new_student = X_test.iloc[[0]].copy()
print("Example input:")
display(new_student)

pred_code = int(best_model.predict(new_student)[0])
pred_label = label_encoder.inverse_transform([pred_code])[0]
print("Predicted outcome:", pred_label)

if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(new_student)[0]
    prob_df = pd.DataFrame({"Class": class_names, "Predicted probability": probabilities})
    display(prob_df)


## 14. Save model, predictions, metrics, and run metadata

In [ ]:
joblib.dump(best_model, OUTPUT_DIR / "best_student_performance_model.joblib")

prediction_df = X_test.copy()
prediction_df["Actual"] = label_encoder.inverse_transform(y_test)
prediction_df["Predicted"] = label_encoder.inverse_transform(y_pred)
prediction_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

metadata = {
    "data_source": DATA_SOURCE,
    "target_column": str(TARGET_COLUMN),
    "class_names": class_names,
    "best_model_by_test_macro_f1": str(best_model_name),
    "random_state": RANDOM_STATE,
    "n_rows_after_target_cleanup": int(len(df)),
    "n_features": int(X.shape[1]),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "warning": "Do not report synthetic demonstration results as real-world research findings."
}
with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print("\nFiles:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)


## 15. Research checklist before publication
- [ ] Verify dataset provenance, permissions, target definition, and data dictionary.
- [ ] Remove leakage features and identifiers.
- [ ] Use training-only cross-validation for model selection.
- [ ] Keep a final test set untouched until evaluation.
- [ ] Compare SHAP/LIME on the same held-out cases where feasible.
- [ ] Define explanation stability mathematically and report repeated trials with uncertainty.
- [ ] Predefine fairness metrics and justify protected/group attributes.
- [ ] Report confidence intervals or variation across seeds, not only one score.
- [ ] Include baselines, limitations, compute environment, and reproducible dependencies.
- [ ] Do not claim novelty or superiority unless experiments support it.
- [ ] Do not publish personal student data.
